In [ ]:
# stress_ray_demo.py – simple Ray workload to light up the dashboard

import os
import time
import ray

print("stress script PID:", os.getpid())

# --- Connect to the existing Ray cluster (or start one if needed) ---
# On Databricks, this typically attaches to the same Ray runtime used by your notebook.
if not ray.is_initialized():
    context = ray.init(
        namespace="databricks_ray_demo",
        ignore_reinit_error=True,
        # address="auto"  # uncomment if connecting to an external Ray cluster
    )
else:
    # When Ray is already running in this process, get_runtime_context()
    # reflects the existing session (no dashboard_url here).
    context = ray.get_runtime_context()

print("Connected to Ray")
print("Cluster resources:", ray.cluster_resources())
print("Available resources:", ray.available_resources())

# Only the object returned by ray.init() has dashboard_url
dashboard_url = getattr(context, "dashboard_url", None)
if dashboard_url:
    print("Dashboard URL:", dashboard_url)
else:
    print("Dashboard URL not available from this context (common on some managed environments).")

# --- Define stress tasks ---
@ray.remote
def cpu_task(n: int = 10_000_000):
    """Simple CPU-bound loop to generate load."""
    x = 0
    for i in range(n):
        x += i % 7
    return x

@ray.remote
def sleepy_task(t: int = 5):
    """Lightweight task that just sleeps for t seconds."""
    time.sleep(t)
    return t

# --- Launch stress workload ---
NUM_CPU_TASKS = 50
NUM_SLEEP_TASKS = 20

print(f"Submitting {NUM_CPU_TASKS} CPU tasks...")
cpu_futures = [cpu_task.remote() for _ in range(NUM_CPU_TASKS)]

print(f"Submitting {NUM_SLEEP_TASKS} sleepy tasks...")
sleepy_futures = [sleepy_task.remote() for _ in range(NUM_SLEEP_TASKS)]

# Wait for completion so object store / task view shows activity
results_cpu = ray.get(cpu_futures)
results_sleep = ray.get(sleepy_futures)

print("Sample CPU results:", results_cpu[:5])
print("Sample Sleep results:", results_sleep[:5])

# Keep cluster busy a bit longer so the dashboard shows activity
HOLD_SECONDS = 15
print(f"Keeping cluster alive for {HOLD_SECONDS}s to view dashboard...")
time.sleep(HOLD_SECONDS)

print("Stress test finished.")

# Optionally: clean shutdown if you want this driver to detach fully
# ray.shutdown()